In [101]:
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

In [102]:
data = pd.read_parquet('dados/dados_tratados/ranking_siconfi.parquet')

data['exercicio'] = data['exercicio'].astype(int)

data = data.sort_values(
    ['municipio', 'exercicio']
)

In [ ]:
estado = sorted(data['estado'].unique())

dropdown_estado = widgets.Dropdown(
    options=estado,
    description='Estados:',
    style={'description_width': 'initial'}
)

municipios = sorted(data['municipio'].unique())

dropdown_municipio = widgets.Dropdown(
    options=[],
    description='Municípios:',
    style={'description_width': 'initial'}
)



In [104]:
saida = widgets.Output()

In [105]:
def atualizar_municipios(change):

    estado_selecionado = str(change['new'])

    municipios = sorted(
        data.loc[
            data['estado'] == estado_selecionado,
            'municipio'
        ].unique()
    )

    # Atualiza as opções do seletor de municípios
    dropdown_municipio.options = municipios

    # Seleciona o primeiro município automaticamente
    if municipios:
        dropdown_municipio.value = municipios[0]
    else:
        dropdown_municipio.value = None

In [106]:
def atualizar_grafico(change=None):

    with saida:
        saida.clear_output(wait=True)

        municipio_selecionado = dropdown_municipio.value

        if not municipio_selecionado:
            return

        dados_municipio = data[
            data['municipio'] == municipio_selecionado
        ].sort_values('exercicio')

        fig = go.Figure()
        
        fig.add_trace(
            go.Scatter(
                x=dados_municipio['exercicio'],
                y=dados_municipio['class_ranking'],
                mode='lines+markers',
                name=municipio_selecionado,
                customdata=dados_municipio[['nota_ranking']],
                hovertemplate=(
                    '<b>%{fullData.name}</b><br>'
                    'Exercício: %{x}<br>'
                    'Ranking: %{y}<br>'
                    'Nota Ranking: %{customdata[0]}'
                    '<extra></extra>'
                )
            )
        )

        fig.update_layout(
            title=f'Evolução da Classificação no Ranking do SICONFI — {municipio_selecionado}',
            xaxis_title='Exercício',
            yaxis_title='Classificação',
            hovermode='closest',
            height=600,
            autosize=True,
            margin=dict(
                l=60,
                r=30,
                t=80,
                b=60
            )
        )

        fig.update_xaxes(
            dtick=1
        )

        # 1º lugar no topo
        fig.update_yaxes(
            autorange='reversed'
        )

        display(fig)

In [107]:
dropdown_estado.observe(
    atualizar_municipios,
    names='value'    
)

dropdown_municipio.observe(
    atualizar_grafico,
    names='value'    
)

In [108]:
atualizar_municipios({
    'new': dropdown_estado.value
})

In [109]:
display(
    widgets.VBox([
        dropdown_estado,
        dropdown_municipio,
        saida
    ])
)